In [1]:
# ===========================
# CONTRADICTION DETECTION WITH CHUNKING + SOFTMAX
# OPTIMIZED: NO DECODE, PRE-CHUNK, NO VERBOSE TRUNCATION, + 2x GPU via DataParallel
# ===========================
!pip install transformers datasets torch networkx pandas openpyxl -q

import pandas as pd
import torch
import numpy as np
import networkx as nx
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ===========================
# STEP 1: GLOBAL CONFIG (easy tuning)
# ===========================

MODEL_NAME = "roberta-large-mnli"

# --- 2-GPU setup ---
USE_FP16 = True  # set False if you see instability (rare for inference)
assert torch.cuda.is_available(), "❌ CUDA not available. You need GPUs for this setup."
assert torch.cuda.device_count() >= 2, f"❌ Need 2 GPUs, found {torch.cuda.device_count()}"

PRIMARY_DEVICE = torch.device("cuda:0")  # DataParallel expects inputs on cuda:0
DEVICE = PRIMARY_DEVICE                 # keep structure consistent

# Chunking params
CHUNK_SIZE = 240
STRIDE = 128
MAX_PAIR_TOKENS = 512

# Batch params (increase to keep both GPUs busy; adjust if OOM)
CHUNK_BATCH_SIZE = 128

# Thresholds
AGREE_THRESHOLD = 0.30
CONTRADICT_CHUNK_THRESH = 0.8

# Files
INPUT_FILE = "/kaggle/input/vidhikarya-fulldataset/Full_answer_Dataset.xlsx"
OUTPUT_FILE = "grouped_responses_chunked.xlsx"

# ===========================
# STEP 2: Load Model + Tokenizer
# ===========================
print("✅ GPUs available:", torch.cuda.device_count())
print("✅ Primary device:", PRIMARY_DEVICE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(PRIMARY_DEVICE)
base_model.eval()

# Wrap with DataParallel to use BOTH GPUs (cuda:0 and cuda:1)
model = torch.nn.DataParallel(base_model, device_ids=[0, 1])
model.eval()
print("✅ Using torch.nn.DataParallel on GPUs:", model.device_ids)

# Label mappings
id2label = {int(k): v.upper() for k, v in base_model.config.id2label.items()}
label2id = {v: k for k, v in id2label.items()}
print("Model label mapping:", id2label)

CONTRA_IDX = label2id.get("CONTRADICTION", 0)
NEUTRAL_IDX = label2id.get("NEUTRAL", 1)
ENTAIL_IDX = label2id.get("ENTAILMENT", 2)

# ===========================
# STEP 3: Helper Functions
# ===========================

def chunk_input_ids(input_ids, chunk_size=CHUNK_SIZE, stride=STRIDE):
    """Split token IDs into overlapping chunks."""
    L = len(input_ids)
    if L == 0:
        return []
    chunks, start = [], 0
    while start < L:
        end = min(start + chunk_size, L)
        chunks.append(input_ids[start:end])
        if end == L:
            break
        start += stride
    return chunks

def tokenize_no_trunc(text: str):
    """Tokenize without truncation/special tokens (so we chunk manually)."""
    return tokenizer.encode(text, add_special_tokens=False)

def build_roberta_pair_ids(a_ids, b_ids, max_len=MAX_PAIR_TOKENS):
    """
    Build RoBERTa pair input IDs directly:
      <s> A </s></s> B </s>
    and ensure total length <= max_len by truncating content if needed.
    """
    cls_id = tokenizer.cls_token_id  # <s>
    sep_id = tokenizer.sep_token_id  # </s>

    max_content = max_len - 4  # total = len(A)+len(B)+4
    if len(a_ids) + len(b_ids) > max_content:
        # simple truncation: fill A then B
        a_keep = min(len(a_ids), max_content)
        b_keep = max_content - a_keep
        a_ids = a_ids[:a_keep]
        b_ids = b_ids[:max(0, b_keep)]

        # ensure some B if A took everything and we still have room
        if b_keep <= 0 and max_content > 0:
            b_keep = min(64, max_content)
            a_keep = max_content - b_keep
            a_ids = a_ids[:max(0, a_keep)]
            b_ids = b_ids[:b_keep]

    input_ids = [cls_id] + a_ids + [sep_id, sep_id] + b_ids + [sep_id]
    attention_mask = [1] * len(input_ids)
    return input_ids, attention_mask

def pad_batch(input_ids_list, attention_mask_list, pad_id, device):
    """Pad variable-length sequences to a batch tensor."""
    max_len = max(len(x) for x in input_ids_list) if input_ids_list else 0
    padded_ids, padded_mask = [], []
    for ids, mask in zip(input_ids_list, attention_mask_list):
        pad_len = max_len - len(ids)
        padded_ids.append(ids + [pad_id] * pad_len)
        padded_mask.append(mask + [0] * pad_len)
    return {
        "input_ids": torch.tensor(padded_ids, dtype=torch.long, device=device),
        "attention_mask": torch.tensor(padded_mask, dtype=torch.long, device=device),
    }

def classify_chunkpairs_probs_from_ids(chunk_pairs_ids, batch_size=CHUNK_BATCH_SIZE):
    """
    Run NLI model on many (chunkA_ids, chunkB_ids) pairs in batches.
    Uses BOTH GPUs via DataParallel. Inputs must be on cuda:0.
    Returns softmax probs [N, 3].
    """
    all_probs = []
    pad_id = tokenizer.pad_token_id

    autocast_ctx = torch.cuda.amp.autocast if USE_FP16 else torch.cpu.amp.autocast  # fallback
    for i in range(0, len(chunk_pairs_ids), batch_size):
        batch = chunk_pairs_ids[i:i+batch_size]

        input_ids_list, attn_list = [], []
        for a_ids, b_ids in batch:
            ids, mask = build_roberta_pair_ids(a_ids, b_ids, max_len=MAX_PAIR_TOKENS)
            input_ids_list.append(ids)
            attn_list.append(mask)

        inputs = pad_batch(input_ids_list, attn_list, pad_id=pad_id, device=PRIMARY_DEVICE)

        with torch.no_grad():
            if USE_FP16:
                with torch.cuda.amp.autocast():
                    logits = model(**inputs).logits
            else:
                logits = model(**inputs).logits

            probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
            all_probs.append(probs)

    if not all_probs:
        return np.zeros((0, len(id2label)), dtype=float)

    return np.vstack(all_probs)

def compute_pair_scores_from_chunks(chunksA, chunksB):
    """Compare two responses given precomputed token-ID chunks."""
    if not chunksA or not chunksB:
        return {"avg_entail": 0.0, "max_entail": 0.0, "max_contradict": 0.0, "n_pairs": 0}

    chunk_pairs_ids = [(a, b) for a in chunksA for b in chunksB]

    probs = classify_chunkpairs_probs_from_ids(chunk_pairs_ids, batch_size=CHUNK_BATCH_SIZE)
    if probs.shape[0] == 0:
        return {"avg_entail": 0.0, "max_entail": 0.0, "max_contradict": 0.0, "n_pairs": 0}

    p_entail = probs[:, ENTAIL_IDX]
    p_contra = probs[:, CONTRA_IDX]

    return {
        "avg_entail": float(np.mean(p_entail)),
        "max_entail": float(np.max(p_entail)),
        "max_contradict": float(np.max(p_contra)),
        "n_pairs": int(probs.shape[0]),
    }

def group_responses_with_chunking(responses):
    """
    Group responses into majority and contradicting groups.
    Pre-tokenize + pre-chunk ONCE per response per row.
    """
    n = len(responses)
    if n == 1:
        return responses[0].strip(), ""

    # Precompute chunks once
    pre_chunks = []
    for r in responses:
        ids = tokenize_no_trunc(r)
        pre_chunks.append(chunk_input_ids(ids))

    # Pair scores
    pair_scores = {}
    for i in range(n):
        for j in range(i + 1, n):
            pair_scores[(i, j)] = compute_pair_scores_from_chunks(pre_chunks[i], pre_chunks[j])

    # Build graph
    G = nx.Graph()
    G.add_nodes_from(range(n))

    for (i, j), s in pair_scores.items():
        if s["max_contradict"] > CONTRADICT_CHUNK_THRESH:
            G.add_edge(i, j, type="contradict", weight=s["max_contradict"])
        if s["avg_entail"] >= AGREE_THRESHOLD:
            G.add_edge(i, j, type="agree", weight=s["avg_entail"])

    # Majority = largest agree component
    agree_edges = [(u, v) for u, v, d in G.edges(data=True) if d["type"] == "agree"]
    agree_subgraph = nx.Graph()
    agree_subgraph.add_nodes_from(G.nodes())
    agree_subgraph.add_edges_from(agree_edges)

    components = list(nx.connected_components(agree_subgraph))
    if components:
        majority_group = max(components, key=len)
    else:
        # fallback by avg entail sum
        node_scores = {i: 0.0 for i in range(n)}
        for (i, j), s in pair_scores.items():
            node_scores[i] += s["avg_entail"]
            node_scores[j] += s["avg_entail"]
        max_score = max(node_scores.values()) if node_scores else 0.0
        majority_group = {i for i, sc in node_scores.items() if sc >= 0.9 * max_score} if max_score > 0 else {0}

    # Contradictions vs majority
    contradict_group = []
    for node in range(n):
        if node in majority_group:
            continue
        for maj in majority_group:
            if G.has_edge(node, maj) and G[node][maj]["type"] == "contradict":
                contradict_group.append(node)
                break

    majority_responses = " ||| ".join([responses[i].strip() for i in sorted(list(majority_group))])
    contradict_responses = " ||| ".join([responses[i].strip() for i in sorted(contradict_group)])

    return majority_responses, contradict_responses

# ===========================
# STEP 4: Run on Excel
# ===========================
df = pd.read_excel(INPUT_FILE)
assert "Final_answers" in df.columns, "❌ Column 'Final_answers' not found!"

majority_col, contradict_col = [], []

for responses_str in tqdm(df["Final_answers"], desc="Processing questions"):
    if pd.isna(responses_str) or str(responses_str).strip() == "":
        majority_col.append("")
        contradict_col.append("")
        continue

    responses = [r.strip() for r in str(responses_str).split("|||") if r.strip()]
    if not responses:
        majority_col.append("")
        contradict_col.append("")
        continue

    maj, contra = group_responses_with_chunking(responses)
    majority_col.append(maj)
    contradict_col.append(contra)

df["Majority_group"] = majority_col
df["Contradicting_group"] = contradict_col
df.to_excel(OUTPUT_FILE, index=False)
print(f"✅ Done! Results saved to {OUTPUT_FILE}")


✅ GPUs available: 2
✅ Primary device: cuda:0


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.43G [00:00<?, ?B/s]

Some weights of the model checkpoint at roberta-large-mnli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


✅ Using torch.nn.DataParallel on GPUs: [0, 1]
Model label mapping: {0: 'CONTRADICTION', 1: 'NEUTRAL', 2: 'ENTAILMENT'}


Processing questions:   0%|          | 0/37971 [00:00<?, ?it/s]/tmp/ipykernel_55/2036214276.py:155: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Processing questions: 100%|██████████| 37971/37971 [9:29:04<00:00,  1.11it/s]    


✅ Done! Results saved to grouped_responses_chunked.xlsx
